# Customer Churn Analysis

**Tools:** Python, SQL, Power BI, Pandas, Scikit-learn

Identify demographic and behavioural churn patterns, build a predictive model, and translate results into retention actions.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_csv('../data/customer_churn_20000.csv')
df.head()

In [ ]:
print(df.info())
print('Missing values:', df.isna().sum().sum())
print('Duplicates:', df.duplicated().sum())
print('Overall churn rate:', round(df['Churn'].mean()*100, 2), '%')

In [ ]:
display(df.groupby('Contract').agg(Customers=('Customer_ID','count'), Churn_Rate=('Churn','mean')).assign(Churn_Rate=lambda x:x.Churn_Rate*100).sort_values('Churn_Rate',ascending=False))
display(df.groupby('Internet_Service').Churn.mean().mul(100).sort_values(ascending=False))
display(df.groupby('Satisfaction_Score').Churn.mean().mul(100))

In [ ]:
X = df.drop(columns=['Customer_ID','Churn'])
y = df['Churn']
cat = ['Contract','Internet_Service']
num = [c for c in X.columns if c not in cat]
pre = ColumnTransformer([('cat',OneHotEncoder(handle_unknown='ignore'),cat),('num',StandardScaler(),num)])
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
models = {'Logistic Regression': LogisticRegression(max_iter=1000), 'Random Forest': RandomForestClassifier(n_estimators=180,max_depth=10,random_state=42,n_jobs=-1)}
results=[]
for name,model in models.items():
    pipe=Pipeline([('pre',pre),('model',model)])
    pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test)
    prob=pipe.predict_proba(X_test)[:,1]
    results.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'ROC_AUC':roc_auc_score(y_test,prob)})
pd.DataFrame(results).sort_values('ROC_AUC',ascending=False)

## Results & Findings

- Month-to-month contracts are the highest-risk segment.
- Lower satisfaction is associated with higher churn.
- Repeated support interactions increase churn risk.
- Random Forest is used as the primary retention-risk model because it captures nonlinear interactions among behavioural variables.
- Power BI should surface these risk segments for retention teams.

## Power BI Output

KPI cards: Overall Churn %, Customers Churned, Model ROC-AUC, Average Monthly Charges.

Charts: Churn by Contract, Churn vs Satisfaction, Churn by Internet Service, Churn by Support Calls and Churn by Tenure.

Slicers: Contract, Internet Service, Satisfaction Score, Tenure and Age.